In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor # XG boost

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Pour l'Encoding

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Pour l'evaluation
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


df = pd.read_csv('/Users/bigkems/Desktop/Projets Perso/Formation AI-engineer/Projet Fitness_market/data/clean_gym_data.csv')

df["gym_memberships_log"] = np.log1p(df["gym_memberships"])

# definition de la target
y = df["gym_memberships_log"]

# definition des Features
X = df.drop(columns=["gym_memberships",
    "gym_memberships_log",
    "gym_penetration_rate",
    "fitness_participation_rate",
    "total_health_club_revenue_usd"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42)

# L'encoding 
 
# Identification des colonnes categorielles
categorical_features = ["country", "region"]

# Creer le transformer 
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)



In [6]:
# L'entrainement du model

model = LinearRegression()
model.fit(X_train_encoded, y_train)

# Les predictions

y_pred = model.predict(X_test_encoded)

In [7]:
# L'evaluation

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R2: 0.4685235134088671
MAE: 1.014193260713591
RMSE: 1.236886875431612


In [8]:
# R2: il predit 47% de la variances, modèle moyen / baseline correct mais faible
# MAE: 1.014193260713591    erreur moyenne ≈ 1 unité sur la variable log, Donc ton modèle se trompe “assez souvent”.
#RMSE: 1.236886875431612  pénalise les grosses erreurs, tu as des prédictions très mauvaises sur certains points

# Le model est limite car trop simple, on devrait utiliser XG boost car ces modeles captent les interactions complexes, la non-linearite, importance variable par pays

In [10]:
from xgboost import XGBRegressor

# Creation du model
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Entrainemeng du model
xgb_model.fit(X_train_encoded, y_train)

# Prediction du model
y_pred_xgb = xgb_model.predict(X_test_encoded)

In [11]:
# Evaluation

print("R2:", r2_score(y_test, y_pred_xgb))
print("MAE:", mean_absolute_error(y_test, y_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))

R2: 0.998602700297235
MAE: 0.0430170225872214
RMSE: 0.0634209490877671


In [ ]:
# Tres bon resultat, voir trop bon... R2 > 99% donc possible data leakage

In [11]:
# TRAIN
y_pred_train = xgb_model.predict(X_train_encoded)

print("TRAIN R2:", r2_score(y_train, y_pred_train))
print("TRAIN MAE:", mean_absolute_error(y_train, y_pred_train))
print("TRAIN RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))

TRAIN R2: 0.9994872257806673
TRAIN MAE: 0.02693966271877737
TRAIN RMSE: 0.03818895532333566


In [12]:
# TEST
y_pred_test = xgb_model.predict(X_test_encoded)

print("TEST R2:", r2_score(y_test, y_pred_test))
print("TEST MAE:", mean_absolute_error(y_test, y_pred_test))
print("TEST RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

TEST R2: 0.998602700297235
TEST MAE: 0.0430170225872214
TEST RMSE: 0.0634209490877671


In [12]:
# cross validation

from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    xgb_model,
    X_train_encoded,
    y_train,
    cv=5,
    scoring="r2"
)

print("CV Scores:", cv_scores)
print("Mean CV:", cv_scores.mean())
print("Std CV:", cv_scores.std())


CV Scores: [0.99863059 0.99724652 0.99797337 0.99819955 0.99869812]
Mean CV: 0.9981496284545649
Std CV: 0.0005256113645897941


In [13]:
# Randomisation test

from sklearn.utils import shuffle

# Mélanger la target
y_random = shuffle(y_train, random_state=42)

# Nouveau modèle
random_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Entraînement sur target mélangée
random_model.fit(X_train_encoded, y_random)

# Prediction
y_random_pred = random_model.predict(X_test_encoded)

# Evaluation
print("Randomized R2:", r2_score(y_test, y_random_pred))

Randomized R2: -0.041562579697839386
